# Segmentación Socio-Demográfica y Conductual: K-Means y LCA (StepMix)
## Mercado: E-commerce global — Clientes Activos

Este notebook presenta un análisis de segmentación avanzado y unificado que combina **K-Means Clustering** y **Análisis de Clases Latentes (LCA / StepMix)** sobre la base de **clientes activos** (`churned == 0`).

Utilizamos las 7 variables categóricas clave identificadas en el análisis sociodemográfico para entrenar el modelo probabilístico y lo contrastamos con una solución clásica de K-Means basada en distancias euclídeas en un espacio mixto estandarizado.

### Variables del Modelo LCA:
| Variable | Tipo StepMix | Justificación |
|---|---|---|
| `gender` | categorical | Diferencia preferencias y patrones de compra |
| `age_group` | categorical | Ciclo de vida del consumidor |
| `region` | categorical | Contexto cultural y de mercado |
| `membership_tier` | categorical | Nivel de relación con la plataforma |
| `preferred_device` | categorical | Canal digital preferido |
| `acquisition_channel` | categorical | Cómo llegó al mercado |
| `preferred_category` | categorical | Segmento de producto de interés |

**Fuentes:** `customers.csv` y `orders.csv`

## 0. Instalación y Librerías

In [ ]:
# Habilite esta línea si no tiene instaladas las librerías necesarias:
# !pip install stepmix scikit-learn pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

try:
    from IPython.display import display
except ImportError:
    display = print

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score, davies_bouldin_score, calinski_harabasz_score
from stepmix.stepmix import StepMix

%matplotlib inline

# Paleta corporativa premium
PALETTE = ['#2E86AB', '#F18F01', '#C73E1D', '#44BBA4', '#A23B72', '#393E41']
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42
print('Librerías cargadas ✓')

## 1. Carga y Preparación de Datos (Filtrado de Clientes Activos)

In [ ]:
cust = pd.read_csv('customers.csv')

# Solo clientes activos (no churned)
cust_active = cust[cust['churned'] == 0].copy().reset_index(drop=True)
print(f'Total Clientes en base original: {len(cust):,}')
print(f'Clientes activos (filtrado): {len(cust_active):,}')

# Fecha de referencia para antigüedad
REFERENCE_DATE = pd.Timestamp('2026-06-01')
cust_active['registration_date'] = pd.to_datetime(cust_active['registration_date'])
cust_active['account_age_days']  = (REFERENCE_DATE - cust_active['registration_date']).dt.days

# Grupo etario (variable ordinal categórica)
cust_active['age_group'] = pd.cut(
    cust_active['age'],
    bins=[17, 25, 35, 45, 55, 75],
    labels=['18-25', '26-35', '36-45', '46-55', '56+']
).astype(str)

# Región geográfica
region_map = {
    'United States':'Norteamérica','Canada':'Norteamérica','Mexico':'Norteamérica',
    'United Kingdom':'Europa','Germany':'Europa','France':'Europa',
    'Spain':'Europa','Italy':'Europa','Netherlands':'Europa',
    'Australia':'Oceanía','New Zealand':'Oceanía',
    'India':'Asia','Japan':'Asia','China':'Asia',
    'South Korea':'Asia','Singapore':'Asia',
    'Brazil':'Latinoamérica','Argentina':'Latinoamérica','Chile':'Latinoamérica',
    'South Africa':'África'
}
cust_active['region'] = cust_active['country'].map(region_map).fillna('Otros')

display(cust_active[['age_group','gender','region','membership_tier',
                      'preferred_device','acquisition_channel',
                      'preferred_category']].head())

## 2. Selección de Variables y Codificación para LCA

In [ ]:
# Variables categóricas para el modelo multinomial
CAT_COLS = [
    'gender', 'age_group', 'region', 'membership_tier',
    'preferred_device', 'acquisition_channel', 'preferred_category'
]

# LabelEncoder por columna: cada variable conserva su propio espacio de categorías
encoders    = {}
cat_mapping = {}
X_parts     = []

for col in CAT_COLS:
    le = LabelEncoder()
    encoded = le.fit_transform(cust_active[col])
    encoders[col]    = le
    cat_mapping[col] = list(le.classes_)
    X_parts.append(encoded.reshape(-1, 1))

# Matriz final de entrada para StepMix
X_lca = np.hstack(X_parts).astype(float)

# Descriptor de medición explícito (dict-of-dicts)
MEASUREMENT = {
    col: {'model': 'categorical', 'n_columns': 1}
    for col in CAT_COLS
}

print(f'Matriz LCA: {X_lca.shape[0]:,} obs × {X_lca.shape[1]} variables')
print('\nCategorías por variable:')
for col, cats in cat_mapping.items():
    print(f'  {col:35s} ({len(cats)} cats): {cats}')

## 3. Selección del Número Óptimo de Clases (Barrido K)

In [ ]:
# Barrido de K = 2 a 6 para encontrar el modelo con menor BIC
resultados = []

for k in range(2, 7):
    print(f'Ajustando modelo con k = {k}...')
    model_k = StepMix(
        n_components = k,
        measurement  = MEASUREMENT,
        random_state = RANDOM_STATE,
        n_init       = 5,
        max_iter     = 300,
        verbose      = 0,
        progress_bar = 0
    )
    model_k.fit(X_lca)
    
    bic = model_k.bic(X_lca)
    aic = model_k.aic(X_lca)
    
    resultados.append({
        'k': k,
        'BIC': bic,
        'AIC': aic
    })

df_sel = pd.DataFrame(resultados)
print('\nTabla completa de Criterios de Información:')
display(df_sel.round(2))

In [ ]:
# Gráfico de selección del modelo (criterio de codo en BIC / mínimo BIC)
plt.figure(figsize=(8, 4.5))
plt.plot(df_sel['k'], df_sel['BIC'], marker='o', color='#C73E1D', linewidth=2, label='BIC (Bayesian Info Criterion)')
plt.plot(df_sel['k'], df_sel['AIC'], marker='s', color='#2E86AB', linewidth=2, linestyle='--', label='AIC (Akaike Info Criterion)')

plt.title('Criterios de Información AIC y BIC según número de clases (LCA)', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Número de clases (k)', fontsize=11)
plt.ylabel('Valor del criterio', fontsize=11)
plt.xticks(df_sel['k'])
plt.legend(frameon=True, shadow=True)
plt.tight_layout()
plt.show()

## 4. Ajuste del Modelo LCA Final ($k=3$)

In [ ]:
# Ajuste del modelo final con k = 3
K_FINAL = 3

lca_final = StepMix(
    n_components = K_FINAL,
    measurement  = MEASUREMENT,
    random_state = RANDOM_STATE,
    n_init       = 10,            # Más reinicializaciones para robustez
    max_iter     = 500,
    verbose      = 0,
    progress_bar = 0
)
lca_final.fit(X_lca)

# Asignación modal (clase con mayor probabilidad)
cust_active['clase_lca']     = lca_final.predict(X_lca)
cust_active['prob_max_clase'] = lca_final.predict_proba(X_lca).max(axis=1)
probs_membresia              = lca_final.predict_proba(X_lca)

# Métricas de ajuste e incertidumbre
bic_f   = lca_final.bic(X_lca)
aic_f   = lca_final.aic(X_lca)
eps     = 1e-12
ent_raw = -np.sum(probs_membresia * np.log(probs_membresia + eps))
ent_rel = 1 - (ent_raw / (len(X_lca) * np.log(K_FINAL)))
avg_prob = cust_active['prob_max_clase'].mean()

print(f'Modelo LCA final ajustado: k = {K_FINAL}')
print(f'  BIC              : {bic_f:,.2f}')
print(f'  AIC              : {aic_f:,.2f}')
print(f'  Entropía norm.   : {ent_rel:.4f}  (1.0 = clases perfectamente separadas)')
print(f'  Prob. media      : {avg_prob:.4f}  (1.0 = asignación de clase 100% segura)')

print('\nTamaño absoluto y relativo de las clases LCA:')
counts = cust_active['clase_lca'].value_counts().sort_index()
for c, n in counts.items():
    print(f'  Clase {c}: {n:,} ({n/len(cust_active)*100:.1f}%)')

## 5. Probabilidades Condicionales (El "ADN" de las Clases)

In [ ]:
# Cálculo empírico de P(categoría | clase)
cond_probs_list = []

for col in CAT_COLS:
    for clase in range(K_FINAL):
        mask = cust_active['clase_lca'] == clase
        vc   = cust_active.loc[mask, col].value_counts(normalize=True)
        for cat, prob in vc.items():
            cond_probs_list.append({
                'variable'    : col,
                'categoría'   : str(cat),
                'clase'       : clase,
                'P(cat|clase)': prob
            })

df_cond = pd.DataFrame(cond_probs_list)

print('=== Categoría más probable por clase (probabilidades condicionales) ===')
for col in CAT_COLS:
    sub   = df_cond[df_cond['variable'] == col]
    pivot = sub.pivot(index='categoría', columns='clase', values='P(cat|clase)').round(3).fillna(0)
    pivot.columns = [f'Clase {c}' for c in pivot.columns]
    print(f'\n── Variable: {col} ──')
    display(pivot)

In [ ]:
# Heatmaps de probabilidades condicionales por variable
n_vars   = len(CAT_COLS)
n_cols_g = 3
n_rows_g = int(np.ceil(n_vars / n_cols_g))
fig, axes = plt.subplots(n_rows_g, n_cols_g, figsize=(16, n_rows_g * 4))
axes_flat = axes.flatten()

for i, col in enumerate(CAT_COLS):
    sub   = df_cond[df_cond['variable'] == col]
    pivot = sub.pivot(index='categoría', columns='clase', values='P(cat|clase)').fillna(0)
    pivot.columns = [f'Clase {c}' for c in pivot.columns]

    sns.heatmap(
        pivot, annot=True, fmt='.2f', cmap='Blues',
        vmin=0, vmax=1, linewidths=0.5,
        ax=axes_flat[i], cbar=False,
        annot_kws={'size': 8.5}
    )
    axes_flat[i].set_title(col, fontweight='bold', fontsize=11)
    axes_flat[i].set_xlabel('')
    axes_flat[i].set_ylabel('')
    axes_flat[i].tick_params(axis='both', labelsize=9)

for j in range(n_vars, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle(
    'Probabilidades Condicionales P(categoría | clase) – LCA de Clientes Activos\n' 
    '(Valores más altos representan características representativas del segmento)',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

## 6. Perfilado Completo y Asignación de Nombres a Clases

In [ ]:
# Perfil numérico medio por clase
perfil_num = cust_active.groupby('clase_lca').agg(
    n_clientes        = ('customer_id', 'count'),
    prob_media        = ('prob_max_clase', 'mean'),
    edad_media        = ('age', 'mean'),
    gasto_total_medio = ('total_spend_usd', 'mean'),
    ordenes_medio     = ('total_orders', 'mean'),
    dias_ult_compra   = ('days_since_last_purchase', 'mean'),
    review_medio      = ('avg_review_score', 'mean'),
    wishlist_medio    = ('wishlist_items', 'mean'),
).round(2)

perfil_num['pct_total'] = (
    perfil_num['n_clientes'] / perfil_num['n_clientes'].sum() * 100
).round(1)

print('=== Perfil Numérico por Clase LCA ===')
display(perfil_num)

print('\n=== Moda de Variables Categóricas por Clase ===')
moda_cat = cust_active.groupby('clase_lca')[CAT_COLS].agg(
    lambda x: x.value_counts().index[0]
)
display(moda_cat)

In [ ]:
# Asignar nombres conceptuales a cada clase LCA según sus características representativas
CLASS_NAMES = {
    0: 'Jóvenes Tecnológicos Emergentes',
    1: 'Mujeres Premium Occidentales',
    2: 'Hombres Estables Multicanal'
}

# Asegurar que todas las clases estén mapeadas, si no usar el valor default
for c in range(K_FINAL):
    if c not in CLASS_NAMES:
        CLASS_NAMES[c] = f'Clase {c}'

cust_active['clase_nombre'] = cust_active['clase_lca'].map(CLASS_NAMES)

print('Mapping definitivo clase → nombre conceptual:')
print(CLASS_NAMES)
print('\nDistribución de clientes por nombre conceptual:')
print(cust_active['clase_nombre'].value_counts())

## 7. Visualizaciones de Perfilado Premium (LCA)

In [ ]:
# 1. Distribución de probabilidades de asignación por clase (certeza de membresía)
plt.figure(figsize=(7, 4.5))
sns.kdeplot(
    data=cust_active, x='prob_max_clase', hue='clase_nombre',
    palette=PALETTE[:K_FINAL], fill=True, common_norm=False, alpha=0.4, linewidth=1.5
)
plt.title('Certeza de Asignación a Clase (Distribución de Probabilidad Máxima)', fontsize=12, fontweight='bold', pad=15)
plt.xlabel('Probabilidad de asignación a clase', fontsize=10)
plt.ylabel('Densidad', fontsize=10)
plt.xlim(0.33, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Heatmap de perfil numérico medio normalizado
vars_heat = ['edad_media', 'gasto_total_medio', 'ordenes_medio',
             'dias_ult_compra', 'review_medio', 'wishlist_medio']

heat_data = perfil_num[vars_heat].copy()
heat_data.index = [CLASS_NAMES[i] for i in heat_data.index]
heat_norm = (heat_data - heat_data.min()) / (heat_data.max() - heat_data.min()).replace(0, 1)

labels_display = ['Edad media', 'Gasto total', 'Órdenes', 'Días últ. compra',
                  'Review medio', 'Wishlist']

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    heat_norm.T,
    annot=heat_data.T.round(1), fmt='g',
    cmap='RdYlGn', linewidths=0.5, ax=ax,
    yticklabels=labels_display,
    cbar_kws={'label': 'Valor normalizado [0,1]'}
)
ax.set_title('Perfil de Clases LCA – Variables Numéricas (Promedios reales anotados)',
             fontweight='bold', fontsize=12, pad=12)
ax.set_xlabel('Clase', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Distribución de variables categóricas clave por clase LCA
vars_barras = [
    ('membership_tier',          'Nivel de Membresía'),
    ('preferred_device',         'Dispositivo Preferido'),
    ('acquisition_channel',      'Canal de Adquisición'),
    ('gender',                   'Género'),
    ('region',                   'Región'),
    ('preferred_category',       'Categoría Preferida'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes_flat = axes.flatten()

for ax, (col, titulo) in zip(axes_flat, vars_barras):
    ct = pd.crosstab(
        cust_active['clase_nombre'],
        cust_active[col],
        normalize='index'
    ) * 100
    ct.plot(kind='bar', ax=ax, color=PALETTE[:ct.shape[1]],
            edgecolor='white', width=0.65)
    ax.set_title(titulo, fontweight='bold', fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('% de Clientes en el Segmento')
    ax.tick_params(axis='x', rotation=12, labelsize=9)
    ax.legend(title='', fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Distribución de Variables Sociodemográficas por Clase LCA', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Proyección PCA 2D para visualización espacial de clases LCA
pca  = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca.fit_transform(X_lca)

fig, ax = plt.subplots(figsize=(8.5, 5.5))
for c in sorted(cust_active['clase_lca'].unique()):
    mask = cust_active['clase_lca'] == c
    ax.scatter(
        X_2d[mask, 0], X_2d[mask, 1],
        s=15, alpha=0.5,
        color=PALETTE[c % len(PALETTE)],
        label=CLASS_NAMES[c]
    )

ax.set_title(f'Separación de Clases LCA en Espacio PCA 2D (k={K_FINAL})', 
             fontweight='bold', fontsize=13, pad=12)
ax.set_xlabel(f'Componente Principal 1 ({pca.explained_variance_ratio_[0]*100:.1f}% var. explicada)', fontsize=10)
ax.set_ylabel(f'Componente Principal 2 ({pca.explained_variance_ratio_[1]*100:.1f}% var. explicada)', fontsize=10)
ax.legend(markerscale=3.5, title='Segmento LCA', loc='best', shadow=True)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Varianza explicada acumulada por 2 componentes principales (PCA): {pca.explained_variance_ratio_.sum()*100:.1f}%')

## 8. Ajuste de K-Means y Comparación de Métricas

Ajustamos un modelo de **K-Means** sobre los mismos clientes activos utilizando exactamente el **mismo conjunto de 7 variables categóricas** representadas como variables dummy (one-hot encoding).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, silhouette_score, davies_bouldin_score, calinski_harabasz_score

# ── K-Means con las mismas variables categóricas que el LCA final ────────────────
cat_cols_km = ['gender', 'age_group', 'region', 'membership_tier', 
               'preferred_device', 'acquisition_channel', 'preferred_category']

X_km_raw = pd.get_dummies(cust_active[cat_cols_km])
X_km     = StandardScaler().fit_transform(X_km_raw)

# Ajuste K-Means final
km_final  = KMeans(n_clusters=K_FINAL, init='k-means++', n_init=10, random_state=RANDOM_STATE)
labels_km = km_final.fit_predict(X_km)
cust_active['clase_kmeans'] = labels_km

# Métricas de Calidad de Clustering
sil_km = silhouette_score(X_km, labels_km, sample_size=2000, random_state=RANDOM_STATE)
db_km  = davies_bouldin_score(X_km, labels_km)
ch_km  = calinski_harabasz_score(X_km, labels_km)

sil_lca = silhouette_score(X_lca, cust_active['clase_lca'].values, sample_size=2000, random_state=RANDOM_STATE)
db_lca  = davies_bouldin_score(X_lca, cust_active['clase_lca'].values)
ch_lca  = calinski_harabasz_score(X_lca, cust_active['clase_lca'].values)

ari = adjusted_rand_score(labels_km, cust_active['clase_lca'].values)

# DataFrame comparativo de métricas
comparacion = pd.DataFrame({
    'Modelo'         : ['K-Means', 'LCA'],
    'k'              : [K_FINAL, K_FINAL],
    'Silhouette ↑'   : [sil_km, sil_lca],
    'Davies-Bouldin ↓': [db_km, db_lca],
    'Calinski-H ↑'   : [ch_km, ch_lca],
    'Entropía ↑'     : ['N/A (No paramétrico)', f'{ent_rel:.4f}'],
    'BIC'            : ['N/A', f'{bic_f:,.1f}'],
}).set_index('Modelo')

print('=== Tabla Comparativa de Calidad: K-Means vs LCA ===')
display(comparacion)
print(f'\nAdjusted Rand Index (Coincidencia/Acuerdo entre soluciones): {ari:.4f}')

In [ ]:
# Heatmap de correspondencia / acuerdo cruzado entre LCA y K-Means
cross = pd.crosstab(
    cust_active['clase_nombre'],
    cust_active['clase_kmeans'],
    rownames=['LCA'],
    colnames=['K-Means'],
    normalize='index'
).round(3) * 100
cross.columns = [f'KM_{c}' for c in cross.columns]

plt.figure(figsize=(7.5, 4))
sns.heatmap(
    cross, annot=True, fmt='.1f', cmap='Blues', 
    linewidths=0.5, cbar_kws={'label': '% de la clase LCA que cae en cada cluster K-Means'}
)
plt.title('Matriz de Coincidencia Cruzada: Clases LCA vs Clusters K-Means\n(Valores expresados en % por Fila)', 
          fontweight='bold', fontsize=12, pad=12)
plt.tight_layout()
plt.show()

## 8.2 Sección de Gráficos de Perfilamiento para K-Means

Para responder al requerimiento de visualizar a fondo los resultados de **K-Means**, creamos un conjunto de visualizaciones equivalentes al del modelo LCA para analizar los clusters generados.

In [ ]:
# 1. Perfil numérico medio por cluster K-Means
perfil_num_km = cust_active.groupby('clase_kmeans').agg(
    n_clientes        = ('customer_id', 'count'),
    edad_media        = ('age', 'mean'),
    gasto_total_medio = ('total_spend_usd', 'mean'),
    ordenes_medio     = ('total_orders', 'mean'),
    dias_ult_compra   = ('days_since_last_purchase', 'mean'),
    review_medio      = ('avg_review_score', 'mean'),
    wishlist_medio    = ('wishlist_items', 'mean'),
).round(2)
perfil_num_km['pct_total'] = (
    perfil_num_km['n_clientes'] / perfil_num_km['n_clientes'].sum() * 100
).round(1)

print('=== Perfil Numérico por Cluster K-Means ===')
display(perfil_num_km)

In [ ]:
# 2. Distribución de tamaño de los clusters K-Means
plt.figure(figsize=(7, 4.5))
sizes_km = cust_active['clase_kmeans'].value_counts(normalize=True).sort_index() * 100
sns.barplot(x=sizes_km.index, y=sizes_km.values, palette='viridis', edgecolor='black', linewidth=1)

plt.title('Distribución de Clientes por Cluster K-Means', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Cluster K-Means', fontsize=11)
plt.ylabel('Porcentaje de Clientes (%)', fontsize=11)
plt.ylim(0, 100)

for i, val in enumerate(sizes_km.values):
    plt.text(i, val + 2, f"{val:.1f}%", ha='center', fontweight='bold', color='darkred', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# 3. Heatmap de perfil numérico medio normalizado por cluster K-Means
heat_data_km = perfil_num_km[['edad_media', 'gasto_total_medio', 'ordenes_medio',
                               'dias_ult_compra', 'review_medio', 'wishlist_medio']].copy()
heat_data_km.index = [f'Cluster KM {i}' for i in heat_data_km.index]
heat_norm_km = (heat_data_km - heat_data_km.min()) / (heat_data_km.max() - heat_data_km.min()).replace(0, 1)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    heat_norm_km.T,
    annot=heat_data_km.T.round(1), fmt='g',
    cmap='RdYlGn', linewidths=0.5, ax=ax,
    yticklabels=['Edad media', 'Gasto total', 'Órdenes', 'Días últ. compra',
                  'Review medio', 'Wishlist'],
    cbar_kws={'label': 'Valor normalizado [0,1]'}
)
ax.set_title('Perfil de Clusters K-Means – Variables Numéricas (Promedios reales anotados)',
             fontweight='bold', fontsize=12, pad=12)
ax.set_xlabel('Cluster K-Means', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Proyección PCA 2D para visualización espacial de clusters K-Means
fig, ax = plt.subplots(figsize=(8.5, 5.5))
for c in sorted(cust_active['clase_kmeans'].unique()):
    mask = cust_active['clase_kmeans'] == c
    ax.scatter(
        X_2d[mask, 0], X_2d[mask, 1],
        s=15, alpha=0.5,
        color=PALETTE[c % len(PALETTE)],
        label=f'Cluster KM {c}'
    )

ax.set_title(f'Separación de Clusters K-Means en Espacio PCA 2D (k={K_FINAL})', 
             fontweight='bold', fontsize=13, pad=12)
ax.set_xlabel(f'Componente Principal 1 ({pca.explained_variance_ratio_[0]*100:.1f}% var. explicada)', fontsize=10)
ax.set_ylabel(f'Componente Principal 2 ({pca.explained_variance_ratio_[1]*100:.1f}% var. explicada)', fontsize=10)
ax.legend(markerscale=3.5, title='Cluster KMeans', loc='best', shadow=True)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 5. Gráfico de burbujas estratégico para K-Means
fig, ax = plt.subplots(figsize=(9, 6.5))

for i, (idx, row) in enumerate(perfil_num_km.iterrows()):
    ax.scatter(
        row['edad_media'], row['gasto_total_medio'],
        s=row['n_clientes'] * 0.05,       # Ponderador de tamaño
        color=PALETTE[i % len(PALETTE)],
        alpha=0.8, edgecolor='black', linewidth=1.5
    )
    ax.annotate(
        f"Cluster KM {idx}\n(n={row['n_clientes']:,} | {row['pct_total']:.1f}%)",
        (row['edad_media'], row['gasto_total_medio']),
        textcoords='offset points', xytext=(12, 5),
        fontsize=9.5, fontweight='bold', arrowprops=dict(arrowstyle="->", color='gray', lw=0.8)
    )

ax.set_xlabel('Edad Media de Clientes (Años)', fontsize=11)
ax.set_ylabel('Gasto Acumulado Medio (USD)', fontsize=11)
ax.set_title('Mapa Estratégico de Posicionamiento de Clusters K-Means\n(Tamaño de la burbuja = volumen relativo de clientes)',
             fontweight='bold', fontsize=13, pad=15)
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()

## 9. Tabla de Posicionamiento Estratégico y Mapa de Clases (LCA)

In [ ]:
# Construimos el Resumen Ejecutivo de Clases LCA para Negocio
resumen = cust_active.groupby('clase_nombre').agg(
    n_clientes      = ('customer_id', 'count'),
    edad_media      = ('age', 'mean'),
    gasto_medio     = ('total_spend_usd', 'mean'),
    ordenes_medios  = ('total_orders', 'mean'),
    prob_asignacion = ('prob_max_clase', 'mean'),
    review_medio    = ('avg_review_score', 'mean'),
).round(2)

resumen['% del mercado'] = (
    resumen['n_clientes'] / resumen['n_clientes'].sum() * 100
).round(1)

for col in ['preferred_device', 'acquisition_channel', 'membership_tier',
            'preferred_category', 'region', 'gender']:
    resumen[f'modal_{col}'] = cust_active.groupby('clase_nombre')[col].agg(
        lambda x: x.value_counts().index[0]
    )

print('=== RESUMEN DE POSICIONAMIENTO DE CLASES LCA ===')
display(resumen.T)

In [ ]:
# Gráfico de burbujas estratégico para LCA: Tamaño × Gasto medio × Edad media
fig, ax = plt.subplots(figsize=(9, 6.5))

for i, (nombre, row) in enumerate(resumen.iterrows()):
    ax.scatter(
        row['edad_media'], row['gasto_medio'],
        s=row['n_clientes'] * 0.05,       # Ponderador de tamaño
        color=PALETTE[i % len(PALETTE)],
        alpha=0.8, edgecolor='black', linewidth=1.5
    )
    ax.annotate(
        f"{nombre}\n(n={row['n_clientes']:,} | {row['% del mercado']:.1f}%)",
        (row['edad_media'], row['gasto_medio']),
        textcoords='offset points', xytext=(12, 5),
        fontsize=9.5, fontweight='bold', arrowprops=dict(arrowstyle="->", color='gray', lw=0.8)
    )

ax.set_xlabel('Edad Media de Clientes (Años)', fontsize=11)
ax.set_ylabel('Gasto Acumulado Medio (USD)', fontsize=11)
ax.set_title('Mapa Estratégico de Posicionamiento de Clases LCA\n(Tamaño de la burbuja = volumen relativo de clientes)',
             fontweight='bold', fontsize=13, pad=15)
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()